# Lab 7a — RAG, workflow et agent

**ENSIM · 5A ASTRE · De l'IA au LLM agentique · Séance 7**
*Bassem Ben Hamed & Ali Khalfallah*

Ce lab se fait **en autonomie, après la séance 7**. Il valide la séance : chaque exemple numérique des diapositives est recalculé ici, puis on construit un **RAG** sur la documentation FreeRTOS et on compare un **workflow** et un **agent** sur un scénario de maintenance, avec **Qwen3-1.7B** (le modèle de la séance 6).

## Objectifs

À la fin du lab, vous savez :

1. calculer un **cosinus**, découper un document en **extraits** (*chunks*) avec recouvrement, compter les extraits ;
2. calculer un score **TF-IDF**, fusionner deux classements avec **RRF**, mesurer le **recall@k** ;
3. indexer le *FreeRTOS Kernel Book* : **embeddings e5**, index **FAISS**, **BM25**, recherche **hybride** et **reclassement** ;
4. reposer la question de la séance 6 (`xTaskBoostPriorityFromISR()`, une fonction **qui n'existe pas**) **avec** et **sans** RAG, et vérifier que le système répond « absente de la documentation » ;
5. coder à la main un **workflow** (routage fixe) et un **agent** (boucle d'appels d'outils), les comparer en réussite et en tokens ;
6. estimer le **coût en tokens** d'une boucle agentique.

## Prérequis et environnement

- Séances 4 à 6 (embeddings, Transformers, LLM) et notions de PyTorch.
- **Google Colab gratuit, GPU T4** recommandé. Le code choisit le périphérique dans l'ordre CUDA, MPS, CPU. Durée d'exécution : environ 3 minutes et demie sur un Mac M4 Pro (GPU MPS, modèles déjà téléchargés), plus le téléchargement de Qwen3-1.7B (4 Go) ; non mesurée sur un T4.
- `faiss-cpu` et `rank_bm25` sont installés à la demande.

## Plan

| Partie | Contenu |
|---|---|
| 1 | Rappel des formules |
| A | Recalculer les exemples numériques des diapositives |
| B | Expériences : indexation du *Kernel Book*, recherche dense, BM25, hybride, RAG, workflow, agent |
| C | Exercices de validation, vérifiés automatiquement |
| | Corrigé |

> **Données** : *FreeRTOS Kernel Book* (13 chapitres en Markdown, dépôt `FreeRTOS/FreeRTOS-Kernel-Book`, licence CC BY-SA 4.0) et sources du noyau `FreeRTOS-Kernel` (MIT) pour vérifier qu'une fonction n'existe pas. Les outils du scénario de maintenance sont **simulés** et déterministes.

## 1. Rappel des formules

| Notion | Formule |
|---|---|
| Cosinus | $\cos(\mathbf{u},\mathbf{v}) = \dfrac{\mathbf{u}\cdot\mathbf{v}}{\lVert\mathbf{u}\rVert\,\lVert\mathbf{v}\rVert}$ |
| Nombre d'extraits | $n = \left\lceil \dfrac{N-L}{L-O} \right\rceil + 1$ ($N$ : document, $L$ : extrait, $O$ : recouvrement, en tokens) |
| IDF | $\mathrm{idf}(t) = \ln(N/\mathrm{df}_t)$ ; score de $d$ : somme des idf des termes de la requête présents dans $d$ |
| RRF | $\mathrm{RRF}(d) = \sum_{s} \dfrac{1}{k + \mathrm{rang}_s(d)}$, $k = 60$ |
| Recall@k | $\dfrac{\lvert \text{pertinents} \cap \text{top-}k \rvert}{\lvert \text{pertinents} \rvert}$ |
| Fidélité | affirmations étayées par les extraits $/$ affirmations de la réponse |
| Coût d'une boucle d'agent | $e_t = P + \sum_{i<t}(s_i + o_i)$, coût $= \sum_t (e_t + s_t)$ |

### Installation et imports

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # macOS : torch et faiss embarquent chacun une copie d'OpenMP, ce qui plante sans cette ligne
import torch
try:
    import faiss
except ImportError:
    %pip install -q faiss-cpu
    import faiss
faiss.omp_set_num_threads(1)                               # 772 vecteurs : un seul thread suffit
try:
    from rank_bm25 import BM25Okapi
except ImportError:
    %pip install -q rank_bm25
    from rank_bm25 import BM25Okapi

import collections, json, math, os, random, re, tempfile, time, urllib.request, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
T0 = time.time()
CACHE = os.path.join(tempfile.gettempdir(), "lab7a_cache")   # téléchargements, hors du dossier du notebook
os.makedirs(CACHE, exist_ok=True)

def choisir_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")
DEVICE = choisir_device()
DTYPE = torch.float16 if DEVICE.type != "cpu" else torch.float32
print("torch", torch.__version__, "| périphérique :", DEVICE)

torch 2.10.0 | périphérique : mps


## Partie A — Recalculer les exemples des diapositives

### A1. Cosinus et top-$k$

Question $\mathbf{q} = (1, 1, 0)$ « envoyer dans une file depuis une interruption » ; trois extraits $c_1$ (`xQueueSend`), $c_2$ (`xQueueSendFromISR`), $c_3$ (`vTaskDelay`).

In [2]:
def cosinus(u, v):
    u, v = np.asarray(u, dtype=float), np.asarray(v, dtype=float)
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

q = np.array([1, 1, 0])
extraits = {"c1 xQueueSend": [2, 0, 1], "c2 xQueueSendFromISR": [2, 2, 1], "c3 vTaskDelay": [1, 0, 2]}
print(f"||q|| = {np.linalg.norm(q):.3f}")
for nom, c in extraits.items():
    print(f"  {nom:22s} q.c = {int(q @ np.array(c))}  ||c|| = {np.linalg.norm(c):.3f}  cos = {cosinus(q, c):.3f}")
print("top-2 :", sorted(extraits, key=lambda n: -cosinus(q, extraits[n]))[:2])
print("vecteurs normés : le cosinus est un produit scalaire ; e5-small : 384 MAC par extrait")

||q|| = 1.414
  c1 xQueueSend          q.c = 2  ||c|| = 2.236  cos = 0.632
  c2 xQueueSendFromISR   q.c = 4  ||c|| = 3.000  cos = 0.943
  c3 vTaskDelay          q.c = 1  ||c|| = 2.236  cos = 0.316
top-2 : ['c2 xQueueSendFromISR', 'c1 xQueueSend']
vecteurs normés : le cosinus est un produit scalaire ; e5-small : 384 MAC par extrait


### A2. Découper un document

$N = 2\,000$ tokens, $L = 512$, $O = 64$ : pas $= 448$.

In [3]:
N, L, O = 2000, 512, 64
n_ext = math.ceil((N - L) / (L - O)) + 1
fenetres = []
for s in range(0, N, L - O):
    fenetres.append((s, min(s + L, N)))
    if s + L >= N:
        break
print("pas =", L - O, "; n = ceil(%d / %d) + 1 = %d extraits" % (N - L, L - O, n_ext), "; fenêtres :", fenetres)
stockes = sum(e - s for s, e in fenetres)
print(f"tokens stockés : {stockes} pour {N} tokens de document, soit +{100 * (stockes / N - 1):.1f} %")

pas = 448 ; n = ceil(1488 / 448) + 1 = 5 extraits ; fenêtres : [(0, 512), (448, 960), (896, 1408), (1344, 1856), (1792, 2000)]
tokens stockés : 2256 pour 2000 tokens de document, soit +12.8 %


### A3. TF-IDF, BM25 et RRF sur trois documents

$d_1$ = {`xQueueSend`, copie, élément, file}, $d_2$ = {`xQueueSendFromISR`, copie, élément, ISR}, $d_3$ = {`vTaskDelay`, bloque, tâche} ; requête {copie, ISR}.

In [4]:
docs = {"d1": {"xqueuesend", "copie", "élément", "file"},
        "d2": {"xqueuesendfromisr", "copie", "élément", "isr"},
        "d3": {"vtaskdelay", "bloque", "tâche"}}
requete = {"copie", "isr"}
df = {t: sum(t in d for d in docs.values()) for t in requete}
idf = {t: math.log(len(docs) / df[t]) for t in requete}
print("idf :", {t: round(v, 3) for t, v in idf.items()}, "(ln(3/2), ln(3/1))")
for nom, d in docs.items():
    print(f"  score({nom}) = {sum(idf[t] for t in requete if t in d):.3f}")

def rrf(rangs, k=60):
    return sum(1 / (k + r) for r in rangs)
print("\nRRF (k = 60) : dense classe d1, d2, d3 ; BM25 ne renvoie que d2")
print(f"  d1 : 1/61 = {rrf([1]):.4f}   d2 : 1/62 + 1/61 = {rrf([2, 1]):.4f}   d3 : 1/63 = {rrf([3]):.4f}")

idf : {'isr': 1.099, 'copie': 0.405} (ln(3/2), ln(3/1))
  score(d1) = 0.405
  score(d2) = 1.504
  score(d3) = 0.000

RRF (k = 60) : dense classe d1, d2, d3 ; BM25 ne renvoie que d2
  d1 : 1/61 = 0.0164   d2 : 1/62 + 1/61 = 0.0325   d3 : 1/63 = 0.0159


### A4. Recall@k, fidélité, et le coût du prompt augmenté

In [5]:
pertinents = [1, 2, 1, 1]; trouves = [1, 1, 0, 1]
rappels = [t / p for p, t in zip(pertinents, trouves)]
print("recall@3 par question :", rappels, "-> moyenne", np.mean(rappels))
print("fidélité : 3 affirmations étayées sur 4 =", 3 / 4)

P_sys, P_q, K, L_ext = 200, 40, 3, 512
avec = P_sys + P_q + K * L_ext
print(f"\nPrompt augmenté : {P_sys} + {P_q} + {K} x {L_ext} = {avec} tokens ; sans RAG : {P_sys + P_q} ; rapport {avec / (P_sys + P_q):.1f}")

def cout_boucle(P, pas):
    """pas : liste de (s_t, o_t). Renvoie (tokens d'entrée, tokens de sortie, total)."""
    entree = sortie = 0
    historique = 0
    for s, o in pas:
        entree += P + historique
        sortie += s
        historique += s + o
    return entree, sortie, entree + sortie
e, s, tot = cout_boucle(340, [(30, 40), (30, 40), (40, 20), (60, 0)])
print(f"\nAgent de maintenance, 4 appels : entrée {e}, sortie {s}, total {tot} tokens")
wf = 120 + 100 + 60
print(f"Workflow : consigne 120 + observations 100 en entrée, 60 en sortie = {wf} tokens ; rapport agent / workflow = {tot / wf:.1f}")

recall@3 par question : [1.0, 0.5, 0.0, 1.0] -> moyenne 0.625
fidélité : 3 affirmations étayées sur 4 = 0.75

Prompt augmenté : 200 + 40 + 3 x 512 = 1776 tokens ; sans RAG : 240 ; rapport 7.4

Agent de maintenance, 4 appels : entrée 1770, sortie 160, total 1930 tokens
Workflow : consigne 120 + observations 100 en entrée, 60 en sortie = 280 tokens ; rapport agent / workflow = 6.9


## Partie B — Expériences

### B1. Le *FreeRTOS Kernel Book*

On télécharge les 13 chapitres Markdown. **Repli** si GitHub est indisponible : un très court corpus de secours (paraphrases) permet de dérouler le notebook, mais les résultats sont alors peu représentatifs.

In [6]:
URL_BOOK = "https://raw.githubusercontent.com/FreeRTOS/FreeRTOS-Kernel-Book/main/ch{:02d}.md"
SECOURS = {
 1: "# 1 Introduction\n\n## 1.1 Secours\n\nCe corpus de secours est une paraphrase très courte de la documentation FreeRTOS.\n\n"
    "### 4.7.4 The vTaskDelay() API Function\n\nvTaskDelay() place la tâche appelante dans l'état Bloqué pour un nombre fixe de ticks, relatif à l'instant de l'appel.\n\n"
    "### 4.7.5 The vTaskDelayUntil() API Function\n\nvTaskDelayUntil() débloque la tâche à un instant absolu : elle permet une période fixe.\n\n"
    "### 7.7.1 The xQueueSendToBackFromISR() API Function\n\nxQueueSendToBackFromISR() est la version de xQueueSendToBack() utilisable dans une routine d'interruption.\n\n"
    "### 8.3.3 Priority Inheritance\n\nL'héritage de priorité relève temporairement la priorité du détenteur d'un mutex.\n\n"
    "### 13.3 Stack Overflow\n\nUn débordement de pile se détecte avec configCHECK_FOR_STACK_OVERFLOW et un hook.\n\n"
    "### 7.4.3 The xSemaphoreGiveFromISR() API Function\n\nxSemaphoreGiveFromISR() donne un sémaphore depuis une interruption.\n\n"
    "### 6.5.1 The xTimerCreate() API Function\n\nxTimerCreate() crée un timer logiciel.\n\n"
    "### 10.3.3 The vTaskNotifyGiveFromISR() API Function\n\nvTaskNotifyGiveFromISR() notifie une tâche depuis une interruption.\n\n"
    "## 4.8 The Idle Task and the Idle Task Hook\n\nLa tâche Idle s'exécute quand aucune autre tâche n'est prête ; son hook s'appelle à chaque itération."}

def charger_livre():
    chapitres, origine = {}, "FreeRTOS Kernel Book (GitHub)"
    try:
        for i in range(1, 14):
            chemin = os.path.join(CACHE, f"ch{i:02d}.md")
            if not os.path.exists(chemin):
                with urllib.request.urlopen(URL_BOOK.format(i), timeout=60) as r:
                    open(chemin, "wb").write(r.read())
            chapitres[i] = open(chemin, encoding="utf-8").read()
    except Exception as e:
        print("Téléchargement impossible :", type(e).__name__, "- corpus de secours")
        chapitres, origine = dict(SECOURS), "corpus de secours (paraphrases)"
    return chapitres, origine

chapitres, origine_livre = charger_livre()
print(origine_livre, ":", len(chapitres), "chapitres,", f"{sum(len(t) for t in chapitres.values()):,} caractères".replace(",", " "))

tok_e5 = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")
n_tok_livre = sum(len(tok_e5(t, add_special_tokens=False).input_ids) for t in chapitres.values())
print(f"Tout le livre : {n_tok_livre:,} tokens (tokeniseur de e5), contre 32 768 tokens de contexte pour Qwen3-1.7B".replace(",", " "))

FreeRTOS Kernel Book (GitHub) : 13 chapitres, 625 883 caractères


Tout le livre : 167 719 tokens (tokeniseur de e5)  contre 32 768 tokens de contexte pour Qwen3-1.7B


### B2. Découper en extraits

On découpe d'abord par **section** (titres `#`, `##`, `###`), pour garder le titre comme source de citation ; une section de plus de $L = 256$ tokens est découpée en fenêtres de 256 tokens avec un recouvrement $O = 32$ (pas de 224). Les fenêtres sont tracées sur le texte original grâce aux positions des tokens.

In [7]:
L_EXT, O_EXT = 256, 32

def decouper_livre(chapitres):
    extraits = []
    for ch, texte in chapitres.items():
        for section in re.split(r"(?m)^(?=#{1,3} )", texte):
            if not section.strip():
                continue
            titre = section.strip().splitlines()[0].lstrip("# ").strip()
            enc = tok_e5(section, add_special_tokens=False, return_offsets_mapping=True)
            n, pos = len(enc.input_ids), enc.offset_mapping
            debuts = list(range(0, n, L_EXT - O_EXT)) if n > L_EXT else [0]
            for d in debuts:
                f = min(d + L_EXT, n)
                extraits.append({"ch": ch, "titre": titre, "n_tok": f - d,
                                 "texte": section[pos[d][0]:pos[f - 1][1]]})
                if f >= n:
                    break
    for i, e in enumerate(extraits):
        e["id"] = i
    return extraits

extraits_livre = decouper_livre(chapitres)
n_tok_ext = sum(e["n_tok"] for e in extraits_livre)
print(f"{len(extraits_livre)} extraits (L = {L_EXT}, O = {O_EXT}), {n_tok_ext:,} tokens stockés pour {n_tok_livre:,} tokens de livre".replace(",", " "))
print("tokens par extrait : min", min(e["n_tok"] for e in extraits_livre), "médiane", int(np.median([e["n_tok"] for e in extraits_livre])),
      "max", max(e["n_tok"] for e in extraits_livre))
e = extraits_livre[400]
print(f"\nExemple d'extrait #{e['id']} (ch{e['ch']:02d}, « {e['titre']} », {e['n_tok']} tokens) :\n{e['texte'][:350]}...")

876 extraits (L = 256  O = 32)  187 368 tokens stockés pour 167 719 tokens de livre
tokens par extrait : min 5 médiane 256 max 256

Exemple d'extrait #400 (ch06, « 6.5.2 The xTimerStart() API Function », 256 tokens) :
 at which the callback was executed. */
    vPrintStringAndNumber( "One-shot timer callback executing", xTimeNow );

    /* File scope variable. */
    ulCallCount++;
}
```
***Listing 6.6*** *The callback function used by the one-shot timer in Example 6.1*


<a name="list6.7" title="Listing 6.7 The callback function used by the auto-reload timer in...


### B3. Index dense (e5 + FAISS) et index lexical (BM25)

`multilingual-e5-small` (384 dimensions) comprend le français et l'anglais ; il demande les préfixes `query: ` et `passage: `. Les vecteurs sont normalisés : le produit scalaire de l'index FAISS (`IndexFlatIP`) est le cosinus.

In [8]:
enc_e5 = SentenceTransformer("intfloat/multilingual-e5-small", device=DEVICE.type)
t = time.time()
passages = [f"passage: {e['titre']}\n{e['texte']}" for e in extraits_livre]
emb = enc_e5.encode(passages, batch_size=64, normalize_embeddings=True, show_progress_bar=False).astype("float32")
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
print(f"index FAISS : {index.ntotal} vecteurs de dimension {emb.shape[1]}, calculé en {time.time() - t:.0f} s")

def jetons(texte):
    return re.findall(r"[a-zA-Z_][a-zA-Z0-9_]*|\d+", texte.lower())
bm25 = BM25Okapi([jetons(e["titre"] + " " + e["texte"]) for e in extraits_livre])

def recherche_dense(question, k=20):
    v = enc_e5.encode([f"query: {question}"], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(v, k)
    return [int(i) for i in ids[0]], [float(s) for s in scores[0]]

def recherche_bm25(question, k=20):
    s = bm25.get_scores(jetons(question))
    ids = np.argsort(-s)[:k]
    return [int(i) for i in ids], [float(s[i]) for i in ids]

def recherche_hybride(question, k=20):
    fusion = collections.defaultdict(float)
    for classement in (recherche_dense(question, k)[0], recherche_bm25(question, k)[0]):
        for rang, i in enumerate(classement, 1):
            fusion[i] += 1 / (60 + rang)
    ordre = sorted(fusion, key=lambda i: -fusion[i])[:k]
    return ordre, [fusion[i] for i in ordre]

q_test = "Comment envoyer dans une file depuis une interruption ?"
for nom, f in (("dense", recherche_dense), ("BM25", recherche_bm25), ("hybride", recherche_hybride)):
    ids, sc = f(q_test, 3)
    print(f"{nom:8s} :", [(extraits_livre[i]["ch"], extraits_livre[i]["titre"][:45]) for i in ids])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

index FAISS : 876 vecteurs de dimension 384, calculé en 5 s
dense    : [(7, '7.7.1 The xQueueSendToFrontFromISR() and xQue'), (12, '12.2.1 Example configASSERT() definitions'), (5, '5.7.1 The xQueueOverwrite() API Function')]
BM25     : [(2, '2.2.5 FreeRTOS Source Files Common to All Por'), (2, '2.2.5 FreeRTOS Source Files Common to All Por'), (12, '12.2.1 Example configASSERT() definitions')]
hybride  : [(7, '7.7.1 The xQueueSendToFrontFromISR() and xQue'), (2, '2.2.5 FreeRTOS Source Files Common to All Por'), (12, '12.2.1 Example configASSERT() definitions')]


### B4. Évaluer la recherche : recall@$k$

Huit questions en français, annotées **par section** : une section est pertinente si son titre correspond au motif indiqué. Le recall@$k$ est la part des sections pertinentes qui apparaissent dans le top-$k$ des extraits. On compare quatre systèmes : dense, BM25, hybride (top-20 fusionné), et hybride + **reclassement** par un *cross-encoder* (modèle anglais `ms-marco-MiniLM`).

In [9]:
EVAL = [
    ("Quelle est la différence entre vTaskDelay() et vTaskDelayUntil() ?", r"vTaskDelay\(\) API|vTaskDelayUntil\(\) API"),
    ("Comment envoyer dans une file depuis une interruption ?", r"xQueueSendToFrontFromISR|xQueueSendToBackFromISR"),
    ("Qu'est-ce que l'inversion de priorité et comment l'héritage de priorité la limite-t-il ?", r"Priority Inversion|Priority Inheritance"),
    ("Comment détecter un débordement de pile ?", r"Stack Overflow|Stack overflow hook"),
    ("Quand la tâche Idle s'exécute-t-elle et à quoi sert son hook ?", r"Idle Task"),
    ("Comment donner un sémaphore depuis une ISR ?", r"xSemaphoreGiveFromISR"),
    ("Comment créer un timer logiciel ?", r"xTimerCreate"),
    ("Que fait vTaskNotifyGiveFromISR ?", r"vTaskNotifyGiveFromISR"),
]
sections_par_motif = {m: {(e["ch"], e["titre"]) for e in extraits_livre if re.search(m, e["titre"])} for _, m in EVAL}
for (qq, m) in EVAL:
    print(f"{len(sections_par_motif[m]):2d} section(s) pertinente(s) :", qq[:70])

try:
    reclasseur = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE.type)
except Exception as e:
    reclasseur = None
    print("Reclasseur indisponible :", type(e).__name__)

def recherche_reclassee(question, k=20):
    ids, _ = recherche_hybride(question, 20)
    if reclasseur is None:
        return ids[:k], []
    scores = reclasseur.predict([(question, extraits_livre[i]["titre"] + "\n" + extraits_livre[i]["texte"]) for i in ids])
    ordre = np.argsort(-scores)
    return [ids[j] for j in ordre][:k], [float(scores[j]) for j in ordre][:k]

SYSTEMES = {"dense": recherche_dense, "BM25": recherche_bm25, "hybride": recherche_hybride, "hybride + reclassement": recherche_reclassee}

def recall_at_k_sections(systeme, k):
    rappels = []
    for qq, m in EVAL:
        pertinentes = sections_par_motif[m]
        if not pertinentes:
            continue
        trouvees = {(extraits_livre[i]["ch"], extraits_livre[i]["titre"]) for i in systeme(qq, k)[0]}
        rappels.append(len(pertinentes & trouvees) / len(pertinentes))
    return float(np.mean(rappels))

tab_recall = pd.DataFrame({nom: {f"recall@{k}": recall_at_k_sections(f, k) for k in (3, 10)} for nom, f in SYSTEMES.items()}).T
print("\nMoyenne sur les questions annotées (sections pertinentes retrouvées) :")
print((100 * tab_recall).round(1).to_string())

 1 section(s) pertinente(s) : Quelle est la différence entre vTaskDelay() et vTaskDelayUntil() ?
 1 section(s) pertinente(s) : Comment envoyer dans une file depuis une interruption ?
 2 section(s) pertinente(s) : Qu'est-ce que l'inversion de priorité et comment l'héritage de priorit
 2 section(s) pertinente(s) : Comment détecter un débordement de pile ?
 4 section(s) pertinente(s) : Quand la tâche Idle s'exécute-t-elle et à quoi sert son hook ?
 1 section(s) pertinente(s) : Comment donner un sémaphore depuis une ISR ?
 1 section(s) pertinente(s) : Comment créer un timer logiciel ?
 1 section(s) pertinente(s) : Que fait vTaskNotifyGiveFromISR ?


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Moyenne sur les questions annotées (sections pertinentes retrouvées) :
                        recall@3  recall@10
dense                       78.1       87.5
BM25                        25.0       46.9
hybride                     56.2       84.4
hybride + reclassement      65.6       84.4


**À observer.** Sur ces 8 questions, le dense seul est le meilleur dans l'exécution de l'auteur (recall@3 : 78 %, contre 56 % pour l'hybride et 25 % pour BM25). Les questions sont en français et le livre est en anglais : BM25, purement lexical, ne retrouve guère que les identifiants (`xSemaphoreGiveFromISR`…), ce qui tire probablement l'hybride vers le bas. Le reclasseur, lui aussi anglais, en récupère une partie (66 %). Avec 8 questions, quelques points d'écart ne prouvent rien ; la leçon est de **mesurer sur ses propres questions** avant d'ajouter des briques.

### B5. Charger Qwen3-1.7B

Le même modèle qu'au Lab 6 (Apache 2.0, 4 Go à télécharger la première fois).

In [10]:
MODELE = "Qwen/Qwen3-1.7B"
tok = AutoTokenizer.from_pretrained(MODELE)
try:
    modele = AutoModelForCausalLM.from_pretrained(MODELE, dtype=DTYPE)
except TypeError:                                   # versions plus anciennes de transformers
    modele = AutoModelForCausalLM.from_pretrained(MODELE, torch_dtype=DTYPE)
modele = modele.to(DEVICE).eval()
tok.padding_side = "left"

def generer(messages, max_new_tokens=220, echantillonner=False, temperature=0.7, top_p=0.8, top_k=20, graine=0):
    """Réponse de chat (sans mode raisonnement). Renvoie (texte, tokens d'entrée, tokens de sortie)."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    texte = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    entree = tok(texte, return_tensors="pt").to(DEVICE)
    options = dict(do_sample=True, temperature=temperature, top_p=top_p, top_k=top_k) if echantillonner \
        else dict(do_sample=False, temperature=None, top_p=None, top_k=None)
    torch.manual_seed(graine)
    with torch.no_grad():
        sortie = modele.generate(**entree, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **options)
    gen = sortie[0, entree.input_ids.shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip(), int(entree.input_ids.shape[1]), int(gen.shape[0])

print(generer("Réponds en une phrase : à quoi sert un RTOS ?", 60)[0])

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Un RTOS (Réseau de Contrôle de Système) est un système d'exploitation conçu pour gérer les ressources d'un appareil de manière efficace et fiable, notamment dans les systèmes embarqués.


### B6. La fonction qui n'existe pas : sans RAG, puis avec RAG

La question de la séance 6, sur `xTaskBoostPriorityFromISR()`. **Cette fonction n'existe pas dans FreeRTOS** : on le vérifie d'abord dans le livre. Puis on interroge le modèle **sans** RAG, **avec** RAG (les 3 meilleurs extraits de la recherche hybride), et **avec RAG et garde-fou** : un test déterministe sur les identifiants de la question. Un identifiant (`xTaskBoostPriorityFromISR`) qui n'apparaît dans **aucun** extrait retrouvé est signalé au modèle. Aucun de ces deux mécanismes n'est une garantie : on relit les réponses.

In [11]:
NOM_FAUX = "xTaskBoostPriorityFromISR"
texte_livre = "\n".join(chapitres.values())
print(f"« {NOM_FAUX} » dans le livre :", NOM_FAUX in texte_livre, "| « BoostPriority » :", "BoostPriority" in texte_livre)

QUESTION = f"Que fait la fonction FreeRTOS {NOM_FAUX}() ?"
CONSIGNE = ("Réponds en français, uniquement à partir des extraits. Cite chaque extrait utilisé par son numéro [n]. "
            "Si l'information n'y figure pas, réponds : « absente de la documentation ».")
REFUS = re.compile(r"absente? de la documentation|n'existe pas|inexistant|aucune fonction|pas de fonction|ne figure pas|"
                   r"n'apparaît pas|introuvable", re.I)

def identifiants(texte):
    return sorted(set(re.findall(r"\b[a-z]{1,3}[A-Z][A-Za-z0-9]+\b", texte)))

def prompt_rag(question, k=3, garde_fou=False):
    ids, _ = recherche_hybride(question, k)
    blocs = [f"[{n}] (ch{extraits_livre[i]['ch']:02d}, {extraits_livre[i]['titre']})\n{extraits_livre[i]['texte']}" for n, i in enumerate(ids, 1)]
    contexte = "\n\n".join(blocs)
    remarque = ""
    if garde_fou:
        absents = [x for x in identifiants(question) if x not in contexte]
        if absents:
            remarque = ("\nRemarque du système : " + ", ".join(absents) + " n'apparaît dans aucun des extraits ci-dessus.")
    return [{"role": "system", "content": CONSIGNE},
            {"role": "user", "content": f"Extraits :\n{contexte}{remarque}\n\nQuestion : {question}"}], ids

sans, n_in_sans, n_out_sans = generer(QUESTION, 200)
msgs_rag, ids_rag = prompt_rag(QUESTION)
avec, n_in_rag, n_out_rag = generer(msgs_rag, 200)
msgs_gf, _ = prompt_rag(QUESTION, garde_fou=True)
avec_gf, n_in_gf, n_out_gf = generer(msgs_gf, 200)

print("Extraits retrouvés :", [(f"ch{extraits_livre[i]['ch']:02d}", extraits_livre[i]['titre'][:50]) for i in ids_rag])
for nom, rep, n_in in (("SANS RAG", sans, n_in_sans), ("AVEC RAG", avec, n_in_rag), ("AVEC RAG + garde-fou", avec_gf, n_in_gf)):
    print(f"\n===== {nom} ({n_in} tokens en entrée) | admet l'absence : {bool(REFUS.search(rep))}")
    print(rep[:600])

« xTaskBoostPriorityFromISR » dans le livre : False | « BoostPriority » : False


Extraits retrouvés : [('ch07', '7.2.4 The xHigherPriorityTaskWoken Parameter'), ('ch05', '5.6.5 More Realistic Queue Set Use Cases'), ('ch07', '7.4.3 The xSemaphoreGiveFromISR() API Function')]

===== SANS RAG (26 tokens en entrée) | admet l'absence : False
La fonction `xTaskBoostPriorityFromISR()` est une fonction de **FreeRTOS** qui permet de **booster la priorité d'un tâche (task)** lors d'une **interrupt (interrupt) en cours**.

---

### **Définition de la fonction :**

```c
portBASE_TYPE xTaskBoostPriorityFromISR( xTaskHandle xTaskToBoost, portBASE_TYPE *pxNewPriority );
```

---

### **Fonctionnalité :**

- **Boost la priorité d'une tâche** lors d'une **interrupt**.
- Cela permet de **prioriser la tâche** dans le **système de gestion des tâches (RTOS)**, ce qui peut être utile pour **gérer les interruptions** de manière plus efficace.

---


===== AVEC RAG (775 tokens en entrée) | admet l'absence : True
La fonction `xTaskBoostPriorityFromISR()` est une fonction d'API FreeRTOS qui

**À observer.** Sans RAG, le modèle peut inventer ou refuser, selon le tirage : relisez. Avec RAG, les extraits retrouvés parlent de sujets voisins (ISR, sémaphores, `xHigherPriorityTaskWoken`…), jamais de `xTaskBoostPriorityFromISR` ; la consigne « si l'information n'y figure pas » suffit-elle pour un petit modèle ? Le garde-fou déterministe apporte l'information manquante au modèle. Le **coût** : le prompt augmenté est bien plus long que la question seule.

In [12]:
print(f"Tokens en entrée : sans RAG {n_in_sans}, avec RAG {n_in_rag}  (rapport {n_in_rag / n_in_sans:.1f})")

# Robustesse : plusieurs tirages (T = 0,7) pour chaque configuration
def taux_refus(messages, n=5):
    return sum(bool(REFUS.search(generer(messages, 120, echantillonner=True, graine=g)[0])) for g in range(n))
print("Réponses qui admettent l'absence, sur 5 tirages à T = 0,7 :")
print("  sans RAG             :", taux_refus(QUESTION))
print("  avec RAG             :", taux_refus(msgs_rag))
print("  avec RAG + garde-fou :", taux_refus(msgs_gf))

# Une question dont la réponse EST dans le livre
q_ok = "Quelle est la différence entre vTaskDelay() et vTaskDelayUntil() ?"
msgs_ok, ids_ok = prompt_rag(q_ok)
r_ok, n_in_ok, _ = generer(msgs_ok, 220)
print("\nQuestion avec réponse dans le livre :", q_ok)
print("Extraits :", [(f"ch{extraits_livre[i]['ch']:02d}", extraits_livre[i]['titre'][:45]) for i in ids_ok], f"({n_in_ok} tokens en entrée)")
print(r_ok[:700])

Tokens en entrée : sans RAG 26, avec RAG 775  (rapport 29.8)
Réponses qui admettent l'absence, sur 5 tirages à T = 0,7 :


  sans RAG             : 0


  avec RAG             : 2


  avec RAG + garde-fou : 5



Question avec réponse dans le livre : Quelle est la différence entre vTaskDelay() et vTaskDelayUntil() ?
Extraits : [('ch04', '4.7.5 The vTaskDelayUntil() API Function'), ('ch04', '4.7.5 The vTaskDelayUntil() API Function'), ('ch04', '4.7.5 The vTaskDelayUntil() API Function')] (795 tokens en entrée)
La différence entre `vTaskDelay()` et `vTaskDelayUntil()` réside dans le fait que `vTaskDelay()` spécifie le nombre de ticks d'interrupts qui devront se produire entre la call de `vTaskDelay()` et la reprise du statut *Ready* du task. Cela signifie que le task reste dans le statut *Blocked* jusqu'à ce qu'un nombre spécifié de ticks ait passé depuis la dernière call de `vTaskDelay()`. 

En revanche, `vTaskDelayUntil()` spécifie un temps exact en ticks à partir duquel le task doit être débloqué (passer du statut *Blocked* à *Ready*). Cela permet d'assurer une périodicité fixe, car le temps à partir duquel le task est débloqué est absolu, et non relatif à la call de `vTaskDelay()`.


### B7. Un scénario de maintenance : workflow contre agent

Demande : « La machine M-07 vibre anormalement. Vérifie et ouvre un ticket si nécessaire. » Trois **outils simulés**, déterministes : `lire_capteur(id)`, `historique_maintenance(id)`, `creer_ticket(id, priorite, motif)`.

- **Workflow** : le **code** fixe l'ordre des outils (lecture, historique si le seuil est dépassé, ticket) ; le LLM n'est appelé qu'**une fois**, pour rédiger le compte rendu.
- **Agent** : une **boucle** d'appels d'outils codée à la main ; à chaque tour, le LLM choisit l'outil ou répond. La boucle est bornée (`PAS_MAX`), les appels sont **validés** (outil connu, arguments attendus, priorité autorisée).

In [13]:
class Atelier:
    """Outils simulés : mêmes entrées, mêmes sorties. Compte les créations de ticket."""
    def __init__(self):
        self.tickets = []
    def lire_capteur(self, id):
        return {"vibration_mm_s": 7.8, "seuil_mm_s": 4.5, "temperature_C": 68} if id == "M-07" else {"erreur": "machine inconnue"}
    def historique_maintenance(self, id):
        return {"dernier_remplacement_roulement_jours": 210} if id == "M-07" else {"erreur": "machine inconnue"}
    def creer_ticket(self, id, priorite, motif):
        if priorite not in ("basse", "normale", "haute"):
            return {"erreur": "priorité invalide"}
        self.tickets.append({"id": id, "priorite": priorite, "motif": motif})
        return {"ticket": f"T-{41 + len(self.tickets):04d}"}

DEMANDE = "La machine M-07 vibre anormalement. Vérifie et ouvre un ticket si nécessaire."
SIGNATURES = {"lire_capteur": {"id"}, "historique_maintenance": {"id"}, "creer_ticket": {"id", "priorite", "motif"}}

def workflow(demande=DEMANDE):
    atelier = Atelier(); n_in = n_out = 0
    m = atelier.lire_capteur("M-07")
    obs = [f"lire_capteur : {json.dumps(m)}"]
    if m["vibration_mm_s"] > m["seuil_mm_s"]:
        obs.append(f"historique_maintenance : {json.dumps(atelier.historique_maintenance('M-07'))}")
        obs.append(f"creer_ticket : {json.dumps(atelier.creer_ticket('M-07', 'haute', 'vibration %.1f > %.1f mm/s' % (m['vibration_mm_s'], m['seuil_mm_s'])))}")
    messages = [{"role": "system", "content": "Tu rédiges un compte rendu de maintenance en deux phrases, en français, à partir des observations. N'invente rien."},
                {"role": "user", "content": f"Demande : {demande}\nObservations :\n" + "\n".join(obs)}]
    rapport, n_in, n_out = generer(messages, 120)
    ok = len(atelier.tickets) == 1 and "T-0042" in rapport
    return dict(ok=ok, tickets=len(atelier.tickets), appels_llm=1, tokens_in=n_in, tokens_out=n_out, rapport=rapport)

def extraire_json(texte):
    m = re.search(r"\{.*\}", texte, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

SYSTEME_AGENT = ("Tu es un agent de maintenance. Outils disponibles :\n"
                 "- lire_capteur(id) : dernières mesures d'une machine\n"
                 "- historique_maintenance(id) : historique des interventions\n"
                 "- creer_ticket(id, priorite, motif) : ouvre un ticket (priorite : basse, normale ou haute)\n"
                 "À chaque tour, réponds par UN SEUL objet JSON, sans autre texte : soit un appel "
                 "{\"outil\": \"<nom>\", \"arguments\": {...}}, soit, quand tu as terminé, {\"reponse_finale\": \"<texte>\"}.")
PAS_MAX = 6

def agent(demande=DEMANDE, echantillonner=False, graine=0):
    atelier = Atelier()
    historique = [{"role": "system", "content": SYSTEME_AGENT}, {"role": "user", "content": demande}]
    n_in = n_out = 0; appels = []
    for pas in range(PAS_MAX):
        texte, ei, so = generer(historique, 100, echantillonner=echantillonner, graine=graine * 10 + pas)
        n_in += ei; n_out += so
        appel = extraire_json(texte)
        if appel is None:
            return dict(ok=False, cause="JSON invalide", appels=appels, pas=pas + 1, tokens_in=n_in, tokens_out=n_out, tickets=len(atelier.tickets))
        if "reponse_finale" in appel:
            lu = "lire_capteur" in appels
            ok = lu and len(atelier.tickets) == 1 and atelier.tickets[0]["id"] == "M-07"
            return dict(ok=ok, cause="" if ok else "ticket absent ou multiple / mesure non lue", appels=appels, pas=pas + 1,
                        tokens_in=n_in, tokens_out=n_out, tickets=len(atelier.tickets), reponse=appel["reponse_finale"])
        nom, args = appel.get("outil"), appel.get("arguments", {})
        if nom not in SIGNATURES or not isinstance(args, dict) or set(args) != SIGNATURES[nom]:
            obs = {"erreur": "outil ou arguments invalides"}
        else:
            obs = getattr(atelier, nom)(**args)
        appels.append(nom)
        historique += [{"role": "assistant", "content": texte}, {"role": "user", "content": "Observation : " + json.dumps(obs, ensure_ascii=False)}]
    return dict(ok=False, cause="limite de pas atteinte", appels=appels, pas=PAS_MAX, tokens_in=n_in, tokens_out=n_out, tickets=len(atelier.tickets))

r_wf = workflow()
print("WORKFLOW :", {k: v for k, v in r_wf.items() if k != "rapport"}); print("  rapport :", r_wf["rapport"])
r_ag = agent()
print("\nAGENT (glouton) :", {k: v for k, v in r_ag.items() if k != "reponse"}); print("  réponse :", r_ag.get("reponse"))

WORKFLOW : {'ok': True, 'tickets': 1, 'appels_llm': 1, 'tokens_in': 147, 'tokens_out': 59}
  rapport : La machine M-07 présente une vibration anormale, avec un capteur indiquant une vitesse de vibration de 7,8 mm/s contre un seuil de 4,5 mm/s. Un ticket a été créé : T-0042.



AGENT (glouton) : {'ok': True, 'cause': '', 'appels': ['lire_capteur', 'historique_maintenance', 'creer_ticket'], 'pas': 4, 'tokens_in': 992, 'tokens_out': 127, 'tickets': 1}
  réponse : Le ticket T-0042 a été créé pour la machine M-07 en raison de la vibration anormale. La priorité est haute.


In [14]:
N_ESSAIS = 10 if DEVICE.type != "cpu" else 3
essais_ag = [agent(echantillonner=True, graine=g) for g in range(N_ESSAIS)]
essais_wf = [workflow() for _ in range(3)]                      # le workflow est glouton : on vérifie sa stabilité
print(f"Agent, {N_ESSAIS} exécutions à T = 0,7 :")
print("  réussites :", sum(e["ok"] for e in essais_ag), "sur", N_ESSAIS)
print("  causes d'échec :", dict(collections.Counter(e["cause"] for e in essais_ag if not e["ok"])))
print("  appels LLM par exécution : moyenne %.1f ; tokens entrée %.0f, sortie %.0f" %
      (np.mean([e["pas"] for e in essais_ag]), np.mean([e["tokens_in"] for e in essais_ag]), np.mean([e["tokens_out"] for e in essais_ag])))
print("  séquences d'outils :", dict(collections.Counter(" > ".join(e["appels"]) for e in essais_ag)))
print(f"\nWorkflow, 3 exécutions : réussites {sum(e['ok'] for e in essais_wf)} sur 3 ; 1 appel LLM ; "
      f"tokens entrée {essais_wf[0]['tokens_in']}, sortie {essais_wf[0]['tokens_out']}")
tot_ag = np.mean([e["tokens_in"] + e["tokens_out"] for e in essais_ag]); tot_wf = essais_wf[0]["tokens_in"] + essais_wf[0]["tokens_out"]
print(f"Tokens au total : agent {tot_ag:.0f}, workflow {tot_wf}, rapport {tot_ag / tot_wf:.1f}")

Agent, 10 exécutions à T = 0,7 :
  réussites : 10 sur 10
  causes d'échec : {}
  appels LLM par exécution : moyenne 4.0 ; tokens entrée 992, sortie 128
  séquences d'outils : {'lire_capteur > historique_maintenance > creer_ticket': 10}

Workflow, 3 exécutions : réussites 3 sur 3 ; 1 appel LLM ; tokens entrée 147, sortie 59
Tokens au total : agent 1120, workflow 206, rapport 5.4


In [15]:
# Ordres de grandeur pour une cible (diapositive « Pour un système embarqué ou critique »)
t_in = np.mean([e["tokens_in"] for e in essais_ag]); t_out = np.mean([e["tokens_out"] for e in essais_ag])
for nom, debit in (("Mac M4 Pro, débit mesuré au lab 6", 25.7), ("Jetson Orin Nano, hypothèse du cours", 20.0)):
    print(f"{nom:36s} : {t_out / debit:.1f} s de génération pour {t_out:.0f} tokens à {debit} tokens/s (hors lecture du prompt)")
octets_par_token = 2 * 28 * 8 * 128 * 2          # K et V, 28 couches, 8 têtes de 128, 2 octets en fp16 (lab 6)
print(f"KV-cache de l'agent : {t_in:.0f} tokens x {octets_par_token // 1024} Kio = {t_in * octets_par_token / 1e9:.2f} Go (poids int4 : 0,86 Go)")

Mac M4 Pro, débit mesuré au lab 6    : 5.0 s de génération pour 128 tokens à 25.7 tokens/s (hors lecture du prompt)
Jetson Orin Nano, hypothèse du cours : 6.4 s de génération pour 128 tokens à 20.0 tokens/s (hors lecture du prompt)
KV-cache de l'agent : 992 tokens x 112 Kio = 0.11 Go (poids int4 : 0,86 Go)


**À observer.** Le workflow est prévisible : mêmes appels, même ordre, un seul appel au LLM. L'agent est plus flexible, mais chaque pas renvoie **tout l'historique** (coût en $T^2$) et la réussite dépend du modèle : un petit modèle peut sauter une lecture, appeler deux fois `creer_ticket` ou casser le JSON. Dans l'exécution de l'auteur, l'agent réussit 10 fois sur 10 avec le même enchaînement d'outils : le chemin étant fixe, le workflow suffisait, pour 5 fois moins de tokens. L'agent ne se justifie que si le chemin varie. Pour un système embarqué ou critique : workflow, garde-fous (validation des arguments, bornes, confirmation humaine avant une action irréversible).

## Partie C — Exercices de validation

Chaque fonction ci-dessous renvoie `None` : le vérificateur affiche alors **« À compléter »**. Complétez-les, puis relancez `verifier_tout()`. Le corrigé est à la fin du notebook : essayez d'abord !

In [16]:
def verifier(nom, valeur, attendu, tol=1e-3):
    if valeur is None:
        print(f"{nom} : À compléter")
        return
    try:
        ok = np.allclose(np.asarray(valeur, dtype=float), np.asarray(attendu, dtype=float), atol=tol)
    except Exception:
        ok = False
    if ok:
        print(f"{nom} : Correct")
    else:
        print(f"{nom} : Incorrect (obtenu {np.round(np.asarray(valeur, dtype=float), 4)}, attendu {attendu})")

def verifier_tout():
    verifier("Exercice 1 — cosinus de q et c2", cosinus_ex([1, 1, 0], [2, 2, 1]), 0.9428)
    verifier("Exercice 2 — nombre d'extraits", nb_extraits(2000, 512, 64), 5)
    verifier("Exercice 3 — score idf de d2", score_idf({"copie", "isr"}, {"xqueuesendfromisr", "copie", "élément", "isr"},
                                                     [{"xqueuesend", "copie", "élément", "file"}, {"xqueuesendfromisr", "copie", "élément", "isr"}, {"vtaskdelay", "bloque", "tâche"}]), 1.504)
    verifier("Exercice 4 — RRF de d2 (rangs 2 et 1)", rrf_ex([2, 1]), 0.0325, tol=1e-4)
    verifier("Exercice 5 — recall@3 moyen", recall_moyen([1, 2, 1, 1], [1, 1, 0, 1]), 0.625)
    verifier("Exercice 6 — coût d'une boucle d'agent", cout_agent(340, [(30, 40), (30, 40), (40, 20), (60, 0)]), 1930)

**Exercice 1 — Cosinus.** Complétez `cosinus_ex(u, v)` (la fonction `cosinus` de la partie A est volontairement masquée : refaites-la).

In [17]:
def cosinus_ex(u, v):
    # À compléter
    return None

**Exercice 2 — Nombre d'extraits.** Complétez `nb_extraits(N, L, O)` : $\lceil (N-L)/(L-O) \rceil + 1$ (si $N \leq L$, un seul extrait).

In [18]:
def nb_extraits(N, L, O):
    # À compléter
    return None

**Exercice 3 — Score idf.** Complétez `score_idf(requete, doc, corpus)` : la somme des $\ln(N/\mathrm{df}_t)$ sur les termes $t$ de la requête **présents dans `doc`** ($N$ = nombre de documents du corpus, $\mathrm{df}_t$ = nombre de documents qui contiennent $t$).

In [19]:
def score_idf(requete, doc, corpus):
    # À compléter
    return None

**Exercice 4 — RRF.** Complétez `rrf_ex(rangs, k=60)` : la somme des $1/(k + \text{rang})$ sur les classements où le document apparaît.

In [20]:
def rrf_ex(rangs, k=60):
    # À compléter
    return None

**Exercice 5 — Recall@k.** Complétez `recall_moyen(pertinents, trouves)` : la moyenne, sur les questions, de `trouves[i] / pertinents[i]`.

In [21]:
def recall_moyen(pertinents, trouves):
    # À compléter
    return None

**Exercice 6 — Coût d'une boucle d'agent.** Complétez `cout_agent(P, pas)` : `pas` est une liste de `(s_t, o_t)` ; renvoyez le total de tokens (entrée + sortie) en renvoyant tout l'historique à chaque appel.

In [22]:
def cout_agent(P, pas):
    # À compléter
    return None

verifier_tout()

Exercice 1 — cosinus de q et c2 : À compléter
Exercice 2 — nombre d'extraits : À compléter
Exercice 3 — score idf de d2 : À compléter
Exercice 4 — RRF de d2 (rangs 2 et 1) : À compléter
Exercice 5 — recall@3 moyen : À compléter
Exercice 6 — coût d'une boucle d'agent : À compléter


## Pour aller plus loin

- Changez $L$ et $O$ (128 ou 512 tokens) : le recall@3 monte-t-il ? Le nombre d'extraits et le coût du prompt, eux, évoluent comment ?
- Ajoutez une **question sans réponse** dans le jeu d'évaluation et définissez un seuil de score en dessous duquel le système s'abstient.
- Remplacez le reclasseur anglais par un reclasseur multilingue : le recall@3 de « hybride + reclassement » dépasse-t-il l'hybride seul ?
- Faites passer l'agent en **mode raisonnement** (`enable_thinking=True`) : la réussite augmente-t-elle, et combien de tokens de plus ?
- Ajoutez un quatrième outil piégé (`supprimer_ticket`) et une **confirmation humaine** avant les actions irréversibles.

## Références

- Lewis et al. (2020), *Retrieval-Augmented Generation* ; Robertson et Zaragoza (2009), BM25 ; Cormack et al. (2009), RRF ; Es et al. (2023), RAGAS.
- Yao et al. (2023), ReAct ; Anthropic (2024), *Building effective agents* ; Wang et al. (2024), `multilingual-e5`.

## Corrigé

Essayez d'abord les exercices ! Cette section définit les solutions, puis relance le vérificateur.

In [23]:
def cosinus_ex(u, v):
    u, v = np.asarray(u, dtype=float), np.asarray(v, dtype=float)
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

def nb_extraits(N, L, O):
    return 1 if N <= L else math.ceil((N - L) / (L - O)) + 1

def score_idf(requete, doc, corpus):
    n = len(corpus)
    return sum(math.log(n / sum(t in d for d in corpus)) for t in requete if t in doc)

def rrf_ex(rangs, k=60):
    return sum(1 / (k + r) for r in rangs)

def recall_moyen(pertinents, trouves):
    return float(np.mean([t / p for p, t in zip(pertinents, trouves)]))

def cout_agent(P, pas):
    total, historique = 0, 0
    for s, o in pas:
        total += P + historique + s
        historique += s + o
    return total

verifier_tout()

Exercice 1 — cosinus de q et c2 : Correct
Exercice 2 — nombre d'extraits : Correct
Exercice 3 — score idf de d2 : Correct
Exercice 4 — RRF de d2 (rangs 2 et 1) : Correct
Exercice 5 — recall@3 moyen : Correct
Exercice 6 — coût d'une boucle d'agent : Correct


In [24]:
print(f"Durée d'exécution du notebook : {time.time() - T0:.0f} s")

Durée d'exécution du notebook : 209 s
